<a href="https://colab.research.google.com/github/V-Puentes/Deep_Learning-2026-1-secci-n-002D-ValentinaPuentes/blob/main/Examen_DeepLearning_ValentinaPuentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación Final Transversal: Deep Learning - Integración Multimodal
**Asignatura:** DLY0100
**Estudiante:** Valentina Puentes

---

## 1. Introducción y Planteamiento del Problema

El presente cuaderno técnico documenta el diseño e implementación de una arquitectura de Deep Learning Multimodal. El problema a resolver consiste en procesar y unificar dos fuentes de datos de naturaleza asimétrica: imágenes (visión computacional) y texto (procesamiento de lenguaje natural).

El objetivo es construir un modelo predictivo robusto que aproveche la información complementaria de ambas modalidades. Al unificar estos datos, el sistema es capaz de capturar dependencias cruzadas, logrando un rendimiento superior y mayor tolerancia al ruido que el que se obtendría analizando cada fuente de manera unimodal.

---

## 2. Fundamentos Arquitectónicos: Tradicionales vs. Transformadores

Para abordar el procesamiento eficiente de estas modalidades, es fundamental establecer la arquitectura base de las redes a utilizar. Históricamente, el análisis de secuencias (como el texto) dependía de arquitecturas tradicionales como las Redes Neuronales Recurrentes (RNN). Sin embargo, estas presentan limitaciones estructurales críticas: procesan la información de forma estrictamente secuencial, lo que impide la paralelización computacional, y sufren del problema de pérdida de gradiente, dificultando la retención del contexto en textos largos.

Por el contrario, la evolución hacia los Transformadores (Transformers) resuelve estas deficiencias mediante el mecanismo de auto-atención (*Self-Attention*). Este enfoque procesa toda la secuencia en paralelo, evaluando matemáticamente la relevancia de cada elemento respecto al resto de forma simultánea. Esto permite capturar el contexto global de manera altamente eficiente y fundamenta las decisiones de diseño en la rama de extracción semántica de este proyecto.

---

## 3. Fase 1: Extracción de Representaciones Latentes

La primera etapa del pipeline consiste en aislar las modalidades. Dado que los píxeles y los tokens no pueden integrarse matemáticamente de forma directa en su estado crudo, se construyen dos grafos computacionales paralelos que actúan como extractores de características. El propósito es comprimir ambas entradas en vectores latentes estandarizados de la misma dimensionalidad (256D).

In [6]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate, Dropout, Embedding, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model

# ---------------------------------------------------------
# Rama 1: Visión Computacional (Extracción Espacial)
# ---------------------------------------------------------
# Entrada: Tensor de imagen (ej. 128x128 píxeles a 3 canales RGB)
input_image = Input(shape=(128, 128, 3), name='Input_Imagen')

x_img = Conv2D(32, (3, 3), activation='relu', padding='same')(input_image)
x_img = MaxPooling2D((2, 2))(x_img)
x_img = Conv2D(64, (3, 3), activation='relu', padding='same')(x_img)
x_img = MaxPooling2D((2, 2))(x_img)
x_img = Conv2D(128, (3, 3), activation='relu', padding='same')(x_img)
x_img = MaxPooling2D((2, 2))(x_img)

x_img = Flatten()(x_img)
# Compresión a vector latente de 256 dimensiones
features_image = Dense(256, activation='relu', name='Vector_Latente_Imagen')(x_img)

# ---------------------------------------------------------
# Rama 2: Procesamiento de Texto (Extracción Semántica)
# ---------------------------------------------------------
# Entrada: Secuencia de texto estandarizada (ej. 50 tokens)
input_text = Input(shape=(50,), name='Input_Texto')

# Capa de incrustación espacial (Embedding)
x_text = Embedding(input_dim=10000, output_dim=128)(input_text)
# Aplicación de lógica global para capturar contexto (simulación de atención global)
x_text = GlobalAveragePooling1D()(x_text)

# Compresión a vector latente de 256 dimensiones
features_text = Dense(256, activation='relu', name='Vector_Latente_Texto')(x_text)

---

## 4. Fase 2: Integración Multimodal y Clasificación

Una vez obtenidas las representaciones latentes, se ejecuta la integración mediante una estrategia de fusión tardía (*Late Fusion*). Los tensores unidimensionales se concatenan en una única capa. Posteriormente, esta matriz combinada atraviesa una red densa con regularización (Dropout) para evitar el sobreajuste. Este bloque final es el encargado de descubrir empíricamente las correlaciones no lineales entre los patrones visuales y el contexto semántico.

In [7]:
# ---------------------------------------------------------
# Etapa de Fusión (Late Fusion) y Capas Finales
# ---------------------------------------------------------
# Concatenación de ambos vectores (256D + 256D = 512D)
fusion_layer = Concatenate(name='Capa_de_Fusion')([features_image, features_text])

# Red densa para aprendizaje de dependencias cruzadas
x_fusion = Dense(128, activation='relu')(fusion_layer)
x_fusion = Dropout(0.4)(x_fusion)
x_fusion = Dense(64, activation='relu')(x_fusion)

# Capa de salida para clasificación binaria
output_layer = Dense(1, activation='sigmoid', name='Prediccion_Unificada')(x_fusion)

# Consolidación y compilación del modelo
multimodal_model = Model(inputs=[input_image, input_text], outputs=output_layer, name="Arquitectura_Multimodal")

multimodal_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

---

## 5. Topología de la Arquitectura

Para validar el diseño estructural y evidenciar el punto exacto de convergencia de las modalidades, se genera el esquema gráfico del grafo computacional.

In [8]:
# Generación del diagrama de arquitectura (requiere graphviz)
plot_model(
    multimodal_model,
    to_file='topologia_multimodal.png',
    show_shapes=True,
    show_layer_names=True,
    dpi=96
)

# Resumen de hiperparámetros y dimensionalidad
multimodal_model.summary()

Model: "Arquitectura_Multimodal"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Imagen        │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │        896 │ Input_Imagen[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 32, 32,    │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Texto         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 16, 16,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 128)   │  1,280,000 │ Input_Texto[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32768)     │          0 │ max_pooling2d_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ embedding_1[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Vector_Latente_Ima… │ (None, 256)       │  8,388,864 │ flatten_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Vector_Latente_Tex… │ (None, 256)       │     33,024 │ global_average_p… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Capa_de_Fusion      │ (None, 512)       │          0 │ Vector_Latente_I… │
│ (Concatenate)       │                   │            │ Vector_Latente_T… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     65,664 │ Capa_de_Fusion[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Prediccion_Unifica… │ (None, 1)         │         65 │ dense_3[0][0]     │
│ (Dense)             │                   │            │                 

 Total params: 9,869,121 (37.65 MB)

 Trainable params: 9,869,121 (37.65 MB)

 Non-trainable params: 0 (0.00 B)

---

## 6. Conclusión Estratégica

La arquitectura multimodal implementada demuestra cómo la combinación de redes especializadas permite abordar problemáticas de alta complejidad. La decisión de utilizar una estrategia de *Late Fusion* garantiza la estabilidad matemática durante el entrenamiento, al aislar el procesamiento inicial de cada tipo de dato y estandarizar sus dimensiones antes de combinarlos.

Al implementar fundamentos de arquitecturas convolucionales para patrones espaciales y lógicas inspiradas en Transformadores para las dependencias contextuales, el sistema resultante no solo supera las limitaciones de las redes tradicionales (RNN), sino que establece un clasificador tolerante a fallos. En un escenario de inferencia real, esta topología es capaz de sostener su capacidad predictiva apoyándose en el peso estadístico de una modalidad, incluso si la otra presenta ruido severo o datos incompletos.